# PokerKit Test Notebook

In [1]:

import random

from pokerkit import Automation, NoLimitTexasHoldem

## Play a game of poker (No Limit Texas Hold'em) with 3 players

In [2]:
automations = (
    Automation.ANTE_POSTING,
    Automation.BET_COLLECTION,
    Automation.BLIND_OR_STRADDLE_POSTING,
    Automation.CARD_BURNING,
    Automation.HOLE_CARDS_SHOWING_OR_MUCKING,
    Automation.HAND_KILLING,
    Automation.CHIPS_PUSHING,
    Automation.CHIPS_PULLING,
)

state = NoLimitTexasHoldem.create_state(
    automations=automations,
    ante_trimming_status=True,  # Uniform antes
    raw_antes=0,  # Ante
    raw_blinds_or_straddles=(1, 2),  # Small/Big Blind
    min_bet=2,  # Minimum bet
    raw_starting_stacks=(100, 100, 100),  # Starting stacks
    player_count=3,  # Number of players
)

print(f"Stacks: {state.stacks}")
print(f"Bets: {state.bets}")
print(f"Pot: {state.total_pot_amount}")

Stacks: [99, 98, 100]
Bets: [1, 2, 0]
Pot: 3


In [3]:
# Deal cards
state.deal_hole("AhAd")  # Alice: Pair of Aces
state.deal_hole("KhKd")  # Bob: Pair of Kings
state.deal_hole("QcQd")  # Charlie: Pair of Queens

for i, cards in enumerate(state.hole_cards):
    print(f"Player {i}: {cards}")

Player 0: [Ah, Ad]
Player 1: [Kh, Kd]
Player 2: [Qc, Qd]


In [4]:
print("Current player:", state.actor_index)
print("Can fold:", state.can_fold())
print("Can check/call:", state.can_check_or_call())
print("Can raise:", state.can_complete_bet_or_raise_to())

Current player: 2
Can fold: True
Can check/call: True
Can raise: True


In [5]:
# Charlie calls 2
state.check_or_call()

# Alice calls 1 (already posted SB)
state.check_or_call()

# Bob checks (already posted BB)
state.check_or_call()

print("Stacks:", state.stacks)
print("Pot:", state.total_pot_amount)
print("Current player:", state.actor_index)

Stacks: [98, 98, 98]
Pot: 6
Current player: None


## New game with 4 players

In [6]:
new_state = NoLimitTexasHoldem.create_state(
    automations=automations,
    ante_trimming_status=True,
    raw_antes=0,
    raw_blinds_or_straddles=(5, 10),  # Changed blinds
    min_bet=10,  # Changed minimum bet
    raw_starting_stacks=(500, 500, 500, 500),  # Changed starting stacks
    player_count=4,  # Number of players
)

In [7]:
def random_card():
    ranks = '23456789TJQKA'
    suits = 'cdhs'
    return random.choice(ranks) + random.choice(suits)

In [8]:
ranks = "23456789TJQKA"
suits = "cdhs"

deck = [rank + suit for rank in ranks for suit in suits]

random.shuffle(deck)

print(len(deck))  # 52


52


In [9]:
# Give each player a pair of cards (unique for each player)
for i in range(4):
    # new_state.deal_hole(random_card() + random_card())
    cards = deck.pop(), deck.pop()
    new_state.deal_hole("".join(cards))

for i, cards in enumerate(new_state.hole_cards):
    print(f"Player {i}: {cards}")

Player 0: [Qc, 6d]
Player 1: [2s, Jd]
Player 2: [Qd, Ah]
Player 3: [4c, Ac]


In [10]:
print("Current player:", new_state.actor_index)
print("Stacks:", new_state.stacks)
print("Pot:", new_state.total_pot_amount)

#  Raise to 30 chips
new_state.complete_bet_or_raise_to(30)

Current player: 2
Stacks: [495, 490, 500, 500]
Pot: 15


CompletionBettingOrRaisingTo(commentary=None, player_index=2, amount=30)

In [11]:
print("Stacks:", new_state.stacks)

Stacks: [495, 490, 470, 500]


## Flop

In [12]:
# Deal the flop
state.deal_board("2c7dJs")

print("Board:", state.board_cards)
print("Current player:", state.actor_index)
print("Pot:", state.total_pot_amount)


Board: [[2c], [7d], [Js]]
Current player: 0
Pot: 6


In [13]:
# All three call
# for _ in range(3):
#     print("Acting player:", state.actor_index)
#     state.check_or_call()

In [14]:
# Alice places 10 chips
state.complete_bet_or_raise_to(10)

# Bob calls
state.check_or_call()

# Charlie folds
state.fold()

print("Pot:", state.total_pot_amount)
print("Next actor:", state.actor_index)

Pot: 26
Next actor: None


In [15]:
print("Stacks:", state.stacks)
print("Bets:", state.bets)
print("Board:", state.board_cards)
print("Street index:", state.street_index)

Stacks: [88, 88, 98]
Bets: [0, 0, 0]
Board: [[2c], [7d], [Js]]
Street index: 2


## Turn and River

In [16]:
# Turn
state.deal_board("3h")

print("Board:", state.board_cards)
print("Current player:", state.actor_index)
print("Street:", state.street_index)
print("Pot:", state.total_pot_amount)


Board: [[2c], [7d], [Js], [3h]]
Current player: 0
Street: 2
Pot: 26


In [17]:
# Alice checks
state.check_or_call()

# Bob checks
state.check_or_call()

print("Stacks:", state.stacks)
print("Current player:", state.actor_index)

Stacks: [88, 88, 98]
Current player: None


In [18]:
# River
state.deal_board("9c")

print("Board:", state.board_cards)
print("Current player:", state.actor_index)

Board: [[2c], [7d], [Js], [3h], [9c]]
Current player: 0


In [19]:
state.check_or_call()  # Alice
state.check_or_call()  # Bob

CheckingOrCalling(commentary=None, player_index=1, amount=0)

## Shodown and Payout

In [20]:
print("Game finished:", not state.status)
print("Final stacks:", state.stacks)
print("Payoffs:", state.payoffs)

Game finished: True
Final stacks: [114, 88, 98]
Payoffs: [14, -12, -2]


In [21]:
assert sum(state.stacks) == 300
assert sum(state.payoffs) == 0
assert not state.status